# Distritos y clasificar sensores 

In [7]:
import pandas as pd

df = pd.read_csv("datos/ubicaciones_sensores_trafico.csv", sep=',')

In [8]:
df

,Unnamed: 0,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud
0,0,URB,4.0,3840,01001,Jose Ortega y Gasset E-O - Pº Castellana-Serrano,441615.343346657,4475767.9421385,-3.6883232754856,40.4305018691825
1,1,URB,4.0,3841,01002,Jose Ortega y Gasset O-E - Serrano-Pº Castellana,441705.882339595,4475769.68733175,-3.68725610305613,40.4305239406404
2,2,URB,1.0,3842,01003,Pº Recoletos N-S - Almirante-Prim,441319.371258338,4474841.15423716,-3.6917268449043,40.4221320929972
3,3,URB,4.0,3843,01004,Pº Recoletos S-N - Pl. Cibeles- Recoletos,441301.63298559,4474763.7275141,-3.69192878230734,40.4214333442836
4,4,URB,4.0,3844,01005,(AFOROS) Pº Castellana S-N - Eduardo Dato - P...,441605.765071902,4476132.13924728,-3.68846965033102,40.4337820578943
...,...,...,...,...,...,...,...,...,...,...
5423,5423,URB,16.0,11235,72406,Av. Secundino Zuazo - César Cort Boti-Gta. Jos...,448.388.147.010.537,448.286.747.130.414,-360.905.755.542.047,404.949.062.864.003
5424,5424,URB,16.0,11276,72407,Av. Manuel Fraga Iribarne (CARRIL BUS) - AV. M...,44.843.325.332.997,448.251.275.311.267,-360.849.642.913.409,404.917.136.031.255
5425,5425,URB,16.0,11277,72408,Av. Manuel Fraga Iribarne (CARRIL BUS) - Jordi...,448.436.665.701.945,448.282.732.568.793,-360.848.176.299.328,404.945.476.496.166
5426,5426,URB,16.0,11279,72409,Secundino Zuazo (CARRIL BUS) - Josefina Aldeco...,447.824.682.462.605,448.294.459.287.868,-361.571.283.696.763,404.955.658.049.048


Hay algunos que tienen formato correcto:

-3.6883232754856	40.4305018691825

Otros que tienen formato incorrecto:

-361.660.626.563.366	40.495.526.719.101

## Ajustar formato

In [9]:
# Indices con el formato incorrecto
indices = df[df['latitud'].str.count('\.') == 4].index

In [11]:
# Ajustar los datos de los que tienen el formato mal

def recortar_cadena(cadena):
    return cadena[:-12]

df.loc[indices, 'longitud'] = df.loc[indices, 'longitud'].apply(recortar_cadena)
df.loc[indices, 'latitud'] = df.loc[indices, 'latitud'].apply(recortar_cadena)

df.loc[indices, 'longitud'] = df.loc[indices, 'longitud'].apply(lambda x: x.replace('.', '') if isinstance(x, str) else x)
df.loc[indices, 'latitud'] = df.loc[indices, 'latitud'].apply(lambda x: x.replace('.', '') if isinstance(x, str) else x)

def colocar_punto(cadena):
    if len(cadena) >= 3:
        return cadena[:2] + '.' + cadena[2:]
    else:
        return cadena

df.loc[indices, 'longitud'] = df.loc[indices, 'longitud'].apply(colocar_punto)
df.loc[indices, 'latitud'] = df.loc[indices, 'latitud'].apply(colocar_punto)

In [12]:
df

,Unnamed: 0,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud
0,0,URB,4.0,3840,01001,Jose Ortega y Gasset E-O - Pº Castellana-Serrano,441615.343346657,4475767.9421385,-3.6883232754856,40.4305018691825
1,1,URB,4.0,3841,01002,Jose Ortega y Gasset O-E - Serrano-Pº Castellana,441705.882339595,4475769.68733175,-3.68725610305613,40.4305239406404
2,2,URB,1.0,3842,01003,Pº Recoletos N-S - Almirante-Prim,441319.371258338,4474841.15423716,-3.6917268449043,40.4221320929972
3,3,URB,4.0,3843,01004,Pº Recoletos S-N - Pl. Cibeles- Recoletos,441301.63298559,4474763.7275141,-3.69192878230734,40.4214333442836
4,4,URB,4.0,3844,01005,(AFOROS) Pº Castellana S-N - Eduardo Dato - P...,441605.765071902,4476132.13924728,-3.68846965033102,40.4337820578943
...,...,...,...,...,...,...,...,...,...,...
5423,5423,URB,16.0,11235,72406,Av. Secundino Zuazo - César Cort Boti-Gta. Jos...,448.388.147.010.537,448.286.747.130.414,-3.60905,40.4949
5424,5424,URB,16.0,11276,72407,Av. Manuel Fraga Iribarne (CARRIL BUS) - AV. M...,44.843.325.332.997,448.251.275.311.267,-3.60849,40.4917
5425,5425,URB,16.0,11277,72408,Av. Manuel Fraga Iribarne (CARRIL BUS) - Jordi...,448.436.665.701.945,448.282.732.568.793,-3.60848,40.4945
5426,5426,URB,16.0,11279,72409,Secundino Zuazo (CARRIL BUS) - Josefina Aldeco...,447.824.682.462.605,448.294.459.287.868,-3.61571,40.4955


In [14]:
df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')
df.dtypes

Unnamed: 0      int64
tipo_elem      object
distrito      float64
id              int64
cod_cent       object
nombre         object
utm_x          object
utm_y          object
longitud      float64
latitud       float64
dtype: object

## Dibujar mapa

In [23]:
import folium

mapa = folium.Map(location=[40.41831, -3.70275], zoom_start=12)

df_condicional = df[(df['distrito'] == 10) | (df['distrito'] == 21)]

for index, row in df_condicional.iterrows():    
    folium.Marker(location=[row['latitud'], row['longitud']], popup=row['nombre']).add_to(mapa)

mapa
